# CTAS
**CREATE TABLE AS SELECT**

Instruccion SQL que crea una tabla nueva a partir de un select y materializa los resultados como datos fisicos en storage.

`CREATE TABLE <table_name>`

`[table_options]`

`AS`

`SELECT ...`

![image_1794123478034.png](./image_1794123478034.png "image_1794123478034.png")

- Ejecuta el plan logico del SELECT.
- Infiere el schema del resultado del query.
- Crea tabla fisica y escribe resultados en formato delta.

#Anatomia de un CTAS

![image_1794124195893.png](./image_1794124195893.png "image_1794124195893.png")

### Componente:

CREATE TABLE => Define el objeto de metastore.

USING DELTA => Formeto fisico de almacenamiento.

COMMENT => Comentario descriptivo para la tabla.

PARTITIONED BY => Layout fisico de los datos.

LOCATION => Ubicacion en storage.

AS SELECT => Logica de transformacion.

##Create table tradicional

_Define estructura pero no datos. requiere un INSERT posterior_

> CREATE TABLE users (
> 
>    id INT,
> 
>    name STRING
>     
> )
> 
> INSERT INTO users
> 
> SELECT ...


## CTAS
_Define estructura + datos en una sola operacion. Schema inferido del query._

> CREATE TABLE users
> 
> AS
> 
> SELECT id, name
> 
> FROM raw_users


# Transformaciones en CTAS

CREATE TABLE silver.users

AS

SELECT

user_id,

lower(email) AS email,

to_date(created_at) AS signup_date

FROM bronze.userss

WHERE is_active = true

- **Transformacion de columnas**

        lower(email),to_date(created_at)

- **filtrado**

        WHERE is_active = true

- **Materializacion**

        Tabla resultante escrita de Delta Lake

In [0]:
%sql
create schema if not exists analytics.ventas

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.nyctaxi_trips_clean
USING DELTA
AS
SELECT 
        tpep_pickup_datetime, 
        tpep_dropoff_datetime,

        TIMESTAMPDIFF(
            MINUTE,
            tpep_pickup_datetime,
            tpep_dropoff_datetime
        ) AS trip_duration_minutes,

        trip_distance,

        ROUND(fare_amount/trip_distance, 2) AS fare_per_mile,

        pickup_zip,
        dropoff_zip

FROM
    samples.nyctaxi.trips
WHERE trip_distance > 0
AND fare_amount > 0;

In [0]:
%sql
CREATE TABLE analytics.ventas.nyctaxi_trips_clean
USING DELTA
AS
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    
    TIMESTAMPDIFF(
        MINUTE,
        tpep_pickup_datetime,
        tpep_dropoff_datetime
    ) AS trip_duration_minutes,
    
    trip_distance,
    
    ROUND(fare_amount / trip_distance, 2) AS fare_per_mile,
    
    pickup_zip,
    dropoff_zip
    
FROM samples.nyctaxi.trips
WHERE trip_distance > 0
AND fare_amount > 0;

# Constrains en Delta Lake

_Reglas de integridad sobre los datos_

**No todos los constrains funcionan igual en un RDBMS clasico. Delta Lake dafine dos categorias claras**

Existen dos tipos de constarins:

- Aplicados (Enforced)

 Se validan estrictamente. Delta rechaza filas que los violen.

 ej: CHECK, NOT NULL.

- Informativos

 Sirven para metadata y optimizacion de query planner, pero no se validan estrictamente.

 ej: PRIMARY KEY, FEREIGN KEY, UNIQUE.


# CHECK Constraint

_El constraint enforced mas solido de Delta Lake_


Al crear la tabla:

`CREATE TABLE users(

id INT,

age INT,

CONSTRAINT age_positive

CHECK (age >= 0)

)`

# Añadir despues

`ALTER TABLE users

ADD CONSTRAINT age_positive

CHECK (age >= 0)`

_si intentas insertar:_

`INSERT INTO users VALUES(1,-5)`

**DELTA LAKE lo rechaza**

Para verificar constrains en una tabla:

- DESCRIBE DETAIL users
- SHOW TBLPROPERTIES users

In [0]:
%sql
CREATE TABLE IF NOT EXISTS analytics.ventas.users(
    id INT,
    age INT
);


In [0]:
%sql
ALTER TABLE analytics.ventas.users
ADD CONSTRAINT age_positive CHECK (age >= 0);

In [0]:
%sql
INSERT INTO analytics.ventas.users VALUES(1, 32)

In [0]:
%sql
INSERT INTO analytics.ventas.users VALUES(1, -32)

## DESCRIBE DETAIL y SHOW TBLPROPERTIES
_Verificar los constrains de una tabla_

In [0]:
%sql
DESCRIBE DETAIL analytics.ventas.users

In [0]:
%sql
SHOW TBLPROPERTIES analytics.ventas.users 

# NOT NULL - PRIMARY KEY - FOREING KEY
_Enforced y informational contrains_

In [0]:
%sql
CREATE TABLE IF NOT EXISTS analytics.ventas.users_2(
    id INT,
    email STRING,
    CONSTRAINT pk_users PRIMARY KEY (id)
);

ALTER TABLE analytics.ventas.users_2
ADD CONSTRAINT email_unique UNIQUE (email);
    
INSERT INTO analytics.ventas.users_2 VALUES(
    1, 'user1@email.com'
),(
    2, 'user2@email.com'
),(
    3, 'user3@email.com'
),(
    4, 'user4@email.com'
),(
    5, 'user5@email.com'
),(
    6, 'user6@email.com'
),(
    7, 'user7@email.com'
),(
    8, 'user8@email.com'
),(
    9, 'user9@email.com'
),(
    10, 'user10@email.com'
),(
    11, 'user11@email.com'
)


# UNIQUE Y EJEMPLOS DE CONSTRAINS

![image_1794289215567.png](./image_1794289215567.png "image_1794289215567.png")